In [1]:
!pip install biopython

In [2]:
import os
import pandas as pd
from Bio.PDB import MMCIFParser
from math import sqrt
import itertools
import json

In [4]:
def calculate_distance(atom1, atom2):
    diff = atom1.coord - atom2.coord
    return sqrt((diff ** 2).sum())

In [5]:
def get_distances(chain_A, chain_B, dock_label):
    distances = []
    for atom_A, atom_B in itertools.product(chain_A.get_atoms(), chain_B.get_atoms()):
        dist = calculate_distance(atom_A, atom_B)
        distances.append({
            "atom1_name": atom_A.get_name(),             # Atom name, e.g., "CG2"
            "atom1_residue": atom_A.get_parent().get_resname(),  # Residue name, e.g., "LEU"
            "atom1_residue_num": atom_A.get_parent().get_id()[1],    # Residue sequence number (e.g., 132)
            "atom1_insertion_code": atom_A.get_parent().get_id()[2],    # Insertion code, if present
            "atom1_chain_id": atom_A.get_parent().get_parent().id,  # Chain ID, e.g., "F1"
            "atom2_name": atom_B.get_name(),             # Atom name, e.g., "CG2"
            "atom2_residue": atom_B.get_parent().get_resname(),  # Residue name, e.g., "LEU"
            "atom2_residue_num": atom_B.get_parent().get_id()[1],    # Residue sequence number (e.g., 132)
            "atom2_insertion_code": atom_B.get_parent().get_id()[2],    # Insertion code, if present
            "atom2_chain_id": atom_B.get_parent().get_parent().id,  # Chain ID, e.g., "F1"
            "distance": dist,
            "dock": dock_label
        })
    return sorted(distances, key=lambda x: x["distance"])  # return top 10

In [6]:
# --- Folder structure ---
base_folder = "kailera_published_structures"

# --- MMCIF parser ---
parser = MMCIFParser(QUIET=True)

# --- Collect all distance entries ---
all_distances = []
all_affinities = []
all_confidences = []

In [58]:
# --- Make dictionary of crystals and drug chains ---
drug_chain_dict = {
    '4ZGM_SEMA_GLP1R_ECConly': 'B',
    '7KI0_SEMA_GLP1R': 'P',
    '7RBT_TIRZ_GIPR': 'P',
    '7RGP_TIRZ_GLP1R': 'P',
    '8YW3_RET_GLP1R': 'P',
    '8YW4_RET_GIPR': 'P',
    '8YWF_RET_GLP1R': 'P'
}

# --- Traverse cif files in base folder ---
cif_list = []
for file in os.listdir(base_folder):
    cif_list.append(file)

print(cif_list)

['7KI0_SEMA_GLP1R.cif', '8YWF_RET_GLP1R.cif', '4ZGM_SEMA_GLP1R_ECConly.cif', '7RBT_TIRZ_GIPR.cif', '8YW3_RET_GLP1R.cif', '.ipynb_checkpoints', '8YW4_RET_GIPR.cif', '7RGP_TIRZ_GLP1R.cif']


In [59]:
cif_file = cif_list[3]
cif_stx = cif_file.replace(".cif", "")

cif_path = os.path.join(base_folder, cif_file)

structure = parser.get_structure(cif_stx, cif_path)
model = structure[0]
chains = list(model.get_chains())


In [60]:
chains

[<Chain id=A>,
 <Chain id=B>,
 <Chain id=E>,
 <Chain id=G>,
 <Chain id=N>,
 <Chain id=P>,
 <Chain id=R>]

In [61]:
protein_1 = chains[0]
protein_2 = chains[1]
protein_3 = chains[2]
protein_4 = chains[3]
protein_5 = chains[4]
drug = chains[5]
protein_6 = chains[6]

# Compute and collect distances
all_distances.extend(get_distances(protein_1, drug, cif_stx))
all_distances.extend(get_distances(protein_2, drug, cif_stx))
all_distances.extend(get_distances(protein_3, drug, cif_stx))
all_distances.extend(get_distances(protein_4, drug, cif_stx))
all_distances.extend(get_distances(protein_5, drug, cif_stx))
all_distances.extend(get_distances(protein_6, drug, cif_stx))

In [62]:
# --- Convert to DataFrame ---
distance_df = pd.DataFrame(all_distances)


In [51]:
distance_df[distance_df.distance < 4]

,atom1_name,atom1_residue,atom1_residue_num,atom1_insertion_code,atom1_chain_id,atom2_name,atom2_residue,atom2_residue_num,atom2_insertion_code,atom2_chain_id,distance,dock
1813440,OH,TYR,152,,R,OE2,GLU,3,,P,2.243279,7RGP_TIRZ_GLP1R
1813441,NH1,ARG,380,,R,OD2,ASP,9,,P,2.487039,7RGP_TIRZ_GLP1R
1813442,OH,TYR,69,,R,O,ILE,27,,P,2.490887,7RGP_TIRZ_GLP1R
1813443,OE1,GLN,234,,R,OH,TYR,1,,P,2.642779,7RGP_TIRZ_GLP1R
1813444,NZ,LYS,197,,R,OG1,THR,7,,P,2.732223,7RGP_TIRZ_GLP1R
...,...,...,...,...,...,...,...,...,...,...,...,...
1813592,CB,LEU,141,,R,CD1,TYR,10,,P,3.974235,7RGP_TIRZ_GLP1R
1813593,CD1,LEU,388,,R,CD2,PHE,6,,P,3.984335,7RGP_TIRZ_GLP1R
1813594,C,THR,298,,R,OG,SER,8,,P,3.984917,7RGP_TIRZ_GLP1R
1813595,O,VAL,30,,R,OE1,GLN,19,,P,3.986455,7RGP_TIRZ_GLP1R


In [63]:
distance_df[distance_df.distance < 4][["atom2_residue", "atom2_residue_num"]].drop_duplicates().sort_values("atom2_residue_num").reset_index(drop=True)

,atom2_residue,atom2_residue_num
0,TYR,1
1,AIB,2
2,GLU,3
3,GLY,4
4,THR,5
5,PHE,6
6,THR,7
7,SER,8
8,ASP,9
9,TYR,10


In [13]:
distance_df_condensed = distance_df.groupby('dock').apply(lambda g: pd.Series({
    'percent_hit': 100 * (((g['atom1_residue_num'].between(61, 69) | g['atom1_residue_num'].between(79, 115)) & (g['distance'] < 4)).sum() / len(g))
})).reset_index()

#result_df = distance_df_condensed.merge(affinity_df, on='dock').merge(confidence_df, on='dock')
#result_df[['company', 'patent_ref_number', 'dimer']] = result_df['dock'].str.split('_', expand=True)

/tmp/ipykernel_14638/3799544491.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  distance_df_condensed = distance_df.groupby('dock').apply(lambda g: pd.Series({


In [16]:
distance_df.to_csv('distance_df_kai.csv')